# Cohorte salud mental

El siguiente codigo construira toda la cohorte de salud mental, la intención es determinar cada una de las fuentes e ir construyendo la logica que llevara.

## Fuentes

- AVICENA
- SIVIGILA
- COHORTE ANTERIOR

## Salidas

- Cohorte mes en curso
- Egresos
- Egresos especifico
- Estado de enfermedad
- Medicamentos
- Población EPS
- Total atenciones

In [1]:
# librerias
import json
import os
import logging
import gc
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta


In [2]:
class CalculadoraFechas:
    def __init__(self, fecha_base=None):
        self.base = fecha_base or datetime.now().date()
        self.periodo_obj = (self.base - relativedelta(months=1)).replace(day=1)

    def params(self, fecha_referencia=None):
        obj=fecha_referencia or self.periodo_obj
        return {
            "anio": obj.strftime("%Y"),
            "mes": obj.strftime("%m"),
            "periodo": obj.strftime("%Y%m")
        }

In [3]:
# --- 2. CLASE FÁBRICA: CONFIGURADOR ---
class FabricaConfig:
    def __init__(self, json_path, params_fecha):
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.params_fecha = params_fecha

    def obtener_lectura(self, key, params_dinamicos=None):
        info = self.data[key].copy()
        # Inyecta fechas en la plantilla y la base
        p_actuales=params_dinamicos or self.params_fecha
        ctx = {**info, **p_actuales}
        ruta = info['plantilla'].format(**ctx)
        
        return {
            "filepath_or_buffer": ruta,
            "sep": info['delimiter'],
            "encoding": info['encoding'],
            "usecols": info.get('columnas'),
            "dtype": str,
            "low_memory": False
        }

In [4]:
# --- 3. CLASE ORQUESTADORA: LOGGING Y CARGA ---
class Orquestador:
    def __init__(self, json_path, fecha_base=None):
        self.fechas = CalculadoraFechas(fecha_base)
        self.fabrica = FabricaConfig(json_path, self.fechas.params())
        self._init_log()

    def _init_log(self):
        os.makedirs('logs', exist_ok=True)
        logging.basicConfig(
            filename=f"logs/carga_{datetime.now().strftime('%Y%m%d')}.log",
            level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.log = logging.getLogger()

    def cargar(self, key):
        try:
            info_fuente=self.fabrica.data.get(key,{})
            n_meses=info_fuente.get("meses_historia", 1)
            procesador = self._obtener_procesador(key)
            dfs_a_concatenar=[]

            for i in range(n_meses):

                fecha_iteracion=self.fechas.periodo_obj-relativedelta(months=i)
                params_iteracion=self.fechas.params(fecha_iteracion)
                cfg = self.fabrica.obtener_lectura(key, params_iteracion)

                if os.path.exists(cfg['filepath_or_buffer']):
                    self.log.info(f"Cargando y limpiando: {params_iteracion['periodo']}")
                    df_mes=pd.read_csv(**cfg)

                    if procesador:
                        df_mes=procesador.procesar(df_mes)

                    dfs_a_concatenar.append(df_mes)
                    del df_mes
                    gc.collect()

                else:
                    self.log.warning(f"No encontrado para {key} en periodo {params_iteracion['periodo']}:{cfg['filepath_or_buffer']}")
            if not dfs_a_concatenar:
                return None
            
            if len(dfs_a_concatenar)>1:
                df_final=pd.concat(dfs_a_concatenar, ignore_index=True)
                self.log.info(f" Exito consolidado: {key} | Meses unidos: {len(dfs_a_concatenar)} | Filas: {len(df_final)}")
                del dfs_a_concatenar
                gc.collect()
            else:
                df_final=dfs_a_concatenar[0]
                self.log.info(f"▷ Exito procesando: {key} | Filas {len(df_final)}")
            return df_final
        except Exception as e:
            self.log.error(f'Error procesando {key}: {e}')
            return None
        
    def _obtener_procesador(self, key):
        mapeo={
            "utilizaciones": ProcesadorUtilizaciones()
        }
        return mapeo.get(key)


In [5]:
class ProcesadorUtilizaciones:
    def __init__(self):
        self.dx_cie10_sm=pd.read_csv("G:/.shortcut-targets-by-id/1wT-pRaNOECz6KC5hndveeLOIGY381o4T/Alteryx/Proyectos/53. Salud Mental/02.Parametros/tmp_dx_cie10.csv", sep=";")
        self.dx_cie10_sm=self.dx_cie10_sm[self.dx_cie10_sm['programa']=="salud Mental"]
    def procesar(self, df):
        df_proceso=df.copy()
        df_proceso['Fecha_prestacion']=pd.to_datetime(df_proceso['Fecha_prestacion'], errors='coerce')
        df_proceso=df_proceso.sort_values(by='Fecha_prestacion', ascending=False)
        df_limpio=df_proceso.drop_duplicates(subset=['TIPO_DOC','Numero_Identificacion_Usuario','DX1'], keep='first')
        df_final=pd.merge(df_limpio, self.dx_cie10_sm, left_on='DX1', right_on='CIE10', how='inner')
        df_final=df_final.drop(columns=['CIE10', 'programa'])
        del df_proceso
        del df_limpio
        return df_final



In [6]:
api = Orquestador("scripts/config_rutas.json")

In [ ]:
# flujo 2, compila las utilizaciones ( se requiere las utilizaciones del ultimo año)
df_utilizaciones = api.cargar("utilizaciones")

In [ ]:
# flujo 1, compila los usuarios finales
df_usuarios = api.cargar("usuarios_finales") 


In [ ]:
# flujo 3, compila las autorizaciones ( se requiere las autorizaciones del ultimo año)
df_autorizaciones = api.cargar("autorizaciones")
# flujo 4, medicamentos
df_medicamentos = api.cargar("medicamentos")
df_avicena = api.cargar("avicena")
df_avicena=df_avicena.dropna(subset="SALUD_MENTAL")
df_cie_10= pd.read_excel('01_Docs/CIE10 SALUD MENTAL_Auditoria_1.xlsx', sheet_name="Consolidado Paramétrica")
df_avicena.head()

,FOLIO,FECHA_APERTURA_HC,HORA_CIERRE_HC,SUCURSAL,TIPDOC_PACIENTE,NUMDOC_PACIENTE,TIPO_DIAGNOSTICO_1,NOMBRES_MEDICO,APELLIDOS_MEDICO,TIPO_DOC_MEDICO,...,ESPECIALIDAD,CIUDAD,FECHA_ATENCION_DATE,sexo,edad,cod_municip,Usuario_Pac,Usuario_Compartido,SALUD_MENTAL,ESTADO_ENFERMEDAD
8,167162570,25/02/2026,09,CENTRO MEDICO COLSANITAS EL DORADO,CC,1000003687,IMPRESION DIAGNOSTICA,DEISY JOHANNA,BALDION AMADO,CC,...,ENFERMERIA,BOGOTA D.C.,25/02/2026,NaN,NaN,NaN,NaN,NaN,F419,NaN
15,166213109,11/02/2026,06,CENTRO MEDICO ZONA IN LOCAL 100,CC,1000018396,IMPRESION DIAGNOSTICA,HEYZER IVETTE,ZAMORA CHAVES,CC,...,PSICOLOGIA,BOGOTA D.C.,11/02/2026,NaN,NaN,NaN,NaN,NaN,Z729,CONTROLADO
24,166528277,16/02/2026,10,LAGY PAOLA ACEVEDO CORZO,CC,1000033512,CONFIRMADO REPETIDO,LAGY PAOLA,ACEVEDO CORZO,CC,...,PSIQUIATRIA,BOGOTA D.C.,16/02/2026,NaN,NaN,NaN,NaN,NaN,F432,NaN
38,167128832,24/02/2026,17,CENTRO MEDICO COLSANITAS PREMIUM 108,CC,1000035966,IMPRESION DIAGNOSTICA,NATALIA STEFANIA,GOMEZ SIERRA,CC,...,PSICOLOGIA,BOGOTA D.C.,24/02/2026,NaN,NaN,NaN,NaN,NaN,R458,NaN
40,165830551,04/02/2026,18,CENTRO MEDICO TEUSAQUILLO,CC,1000036257,CONFIRMADO REPETIDO,OSCAR RENE,RANGEL URREA,CC,...,PSIQUIATRIA,BOGOTA D.C.,04/02/2026,NaN,NaN,NaN,NaN,NaN,F121,CONTROLADO


In [ ]:
df_utilizaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93987 entries, 0 to 93986
Data columns (total 4 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Fecha_prestacion               93987 non-null  datetime64[ns]
 1   Numero_Identificacion_Usuario  93987 non-null  object        
 2   TIPO_DOC                       93987 non-null  object        
 3   DX1                            93987 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 2.9+ MB
